# MRMR versus ARFS all-relevant selection

Use MRMR when you have a feature budget. It adds predictors one at a time,
balancing relevance to the target against redundancy with earlier choices.
Use an all-relevant selector such as GrootCV to look for model-usable
signals by comparing each feature's importance with shuffled features.
The all-relevant rule does not set a target column count.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from arfs.feature_selection import GrootCV, MinRedundancyMaxRelevance

rng = np.random.default_rng(42)
X = pd.DataFrame(rng.normal(size=(400, 4)), columns=["signal", "copy", "weak", "noise"])
X["copy"] = X["signal"] + 0.04 * X["copy"]
y = 5 * X["signal"] + 0.7 * X["weak"] + rng.normal(scale=0.4, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [2]:
mrmr = MinRedundancyMaxRelevance(
    n_features_to_select=2, task="regression", n_jobs=1, show_progress=False
).fit(X_train, y_train)
groot = GrootCV(
    objective="rmse",
    n_folds=3,
    n_iter=1,
    n_jobs=1,
    random_state=42,
    lgbm_params={"num_iterations": 40},
).fit(X_train, y_train)

pd.DataFrame(
    [
        ("MRMR", mrmr.selected_features_, mrmr.transform(X_test).shape[1]),
        (
            "GrootCV",
            groot.selected_features_.tolist(),
            groot.transform(X_test).shape[1],
        ),
    ],
    columns=["Method", "Selected columns", "Held-out width"],
)

,Method,Selected columns,Held-out width
0,MRMR,"[signal, weak]",2
1,GrootCV,"[copy, signal, weak]",3


MRMR's `relevance_`, `redundancy_` and `ranking_` describe its greedy
search. GrootCV's `cv_df` stores fold importances and its `sha_cutoff`
gives the shadow threshold. Each selector produces its own set. MRMR's
selected set has no guaranteed subset relationship to GrootCV's.

In [3]:
display(mrmr.ranking_)
display(groot.cv_df[["feature", "Med"]].rename(columns={"Med": "GrootCV median"}))

,mrmr,relevance,redundancy
signal,inf,0.999907,0.00000
weak,193.744967,0.887448,0.00458


,feature,GrootCV median
0,ShadowVar1,0.004770
1,ShadowVar2,0.009857
2,ShadowVar3,0.003325
3,ShadowVar4,0.005899
4,copy,0.392146
5,noise,0.005477
6,signal,0.455029
7,weak,0.126648
